get historical data from yfinance

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

Calculate sigma on twenty day window preceding the start of our five-day window on which we will calculate portfolio returns. Bin obtained sigmas with tiny bin size (equate all sigmas in a bin with the midpoint of the bin). 

In [ ]:
tickers = [
    "SPY",  # State Street SPDR S&P 500 ETF
    "VTI",  # Vanguard Total Stock Market ETF
    "VEA",  # Vanguard FTSE Developed Markets ETF
    "IWM",  # iShares Russell 2000 ETF
    "VWO",  # Vanguard FTSE Emerging Markets ETF
    "VGT",  # Vanguard Information Technology ETF
    "VHT",  # Vanguard Health Care ETF
    "IBB",  # iShares Nasdaq Biotechnology ETF
    "XBI",  # SPDR S&P Biotech ETF
    "QQQ",  # Invesco QQQ Trust
    "PDBC",  # Invesco Optimum Yield Diversified Commodity Strategy ETF
    "VGK",  # Vanguard FTSE Europe ETF
    "FBT",  # First Trust NYSE Arca Biotechnology Index Fund
    "ARKG",  # ARK Genomic Revolution ETF
    "BBH",  # VanEck Biotech ETF
    "IBIT",  # iShares Bitcoin Trust
    "FBTC",  # Fidelity Bitcoin ETF
    "GBTC",  # Grayscale Bitcoin Trust
    "ETHA",  # Ethereum / Ether ETF
    "BTC",  # Bitcoin (spot / crypto)
    "VNQ",  # Vanguard Real Estate ETF
    "SCHH",  # Schwab U.S. REIT ETF
    "XLRE",  # Real Estate Select Sector SPDR Fund
    "IYR",  # iShares U.S. Real Estate ETF
    "USRT",  # iShares U.S. Real Estate (REIT) ETF
    "ICF",  # iShares Cohen & Steers REIT ETF
    "RWR",  # Invesco S&P 500 Equal Weight Real Estate ETF
    "SMH",  # VanEck Semiconductor ETF
    "SOXX",  # iShares Semiconductor ETF
    "SOXL",  # Direxion Daily Semiconductor Bull 3X Shares
    "NVDL",  # NVIDIA/semiconductor related ETF
    "USD",  # U.S. Dollar (currency)
    "XSD",  # SPDR S&P Semiconductor ETF (equal-weight)
    "GRID",  # First Trust NASDAQ Clean Edge Smart Grid Infrastructure ETF
    "NLR",  # VanEck Nuclear Energy ETF
    "ICLN",  # iShares Global Clean Energy ETF
    "TAN",  # Invesco Solar ETF
    "FTGC",  # First Trust Global Tactical Commodity Strategy Fund
    "HGER",  # Germany equity/sector ETF
]  # stopped at technology on https://etfdb.com/

ticker = "SPY"
start_date, end_date = "2016-03-01", "2026-03-01"
df = yf.download(tickers, start=start_date, end=end_date)["Close"]

[*********************100%***********************]  39 of 39 completed


In [ ]:
window_length = 20
res_list = []

for leverage_factor in np.linspace(1.01, 3, 100):
    for ticker in tickers:

        data = df[[ticker]]
        data["log_returns"] = np.log(data[ticker] / data[ticker].shift(1))
        data["realized_vol"] = data[["log_returns"]].rolling(
            window_length
        ).std() * np.sqrt(252)
        data["leveraged_log_returns"] = np.log(
            leverage_factor * (np.exp(data["log_returns"]) - 1) + 1
        )
        data["PnL"] = np.exp(
            data["leveraged_log_returns"].shift(-5).rolling(5).sum()
        ) - np.exp(data["log_returns"].shift(-5).rolling(5).sum())

        data["leverage_factor"] = leverage_factor

        res_list.append(
            data.dropna().reset_index()[["leverage_factor", "realized_vol", "PnL"]]
        )  # address issue with Ticker column of indices

res_df = pd.concat(res_list)

res_df.to_parquet(f"output/historical_data/lev_sigma_std.parquet")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
